# Initial Steps: load packages, files and functions

In [1]:
import json, re, time, itertools
from copy import deepcopy
from pathlib import Path
from datetime import datetime
import os
import pandas as pd
import requests

In [2]:
cwd=os.getcwd()
cwd_Raw_Data_outputs=os.path.join(cwd,'RawData')#heres where we store freezes of the raw data
# cwd_Figures=os.path.join(cwd,'Figures')#figures and code for generating them can go here
cwd_Output=os.path.join(cwd,'Output Dataframes')

In [5]:
def GetAwardAmount(input_String, Lists):
    #function takes three arguments; the Reporter output, the destination where we store results, and an additional list for storing a freeze of the data
    GetResults = input_String.find("\"results\"") #find the part of the output detailing grant award amount, found after the "results" block of the ouput
    ResultsList = input_String[GetResults:].replace("},", "")# each grant's information is separated by curly brackets; splitting along curly brackets divides info from each grant
    ResultsList = (ResultsList.split("{\""))[1:]    #saving the individual grant amount as a an element in a list of grants
    for iGrant in ResultsList:# for each grant returned by the query
        Award_Start = iGrant.find("\"award_amount\":")#find the part detailing award amount
        Award_End = iGrant.find("\"project_start_date\":")#find the part that comes after the award amount
        DirectCost = iGrant.find("\"direct_cost_amt\":")
        direct_End = iGrant.find("\"indirect_cost_amt\":")
        Award_string = iGrant[Award_Start:Award_End].replace(",", "").split(":")[1]# the amount of money for grant will be between the part addressed as award amount and the direct cost amount
        directCost = iGrant[DirectCost:direct_End].replace(",", "").split(':', 1)[1]
        indirectCost = iGrant[direct_End:].replace(",", "").split(':', 1)[1]
        if not Award_string == "null": # for some reason, some grants do not have an award amount stored in NIH Reporter
            Lists[0] = Lists[0] + int(Award_string)
            if not directCost == "null":
                Lists[1]=Lists[1]+int(directCost)
            if not "null" in indirectCost:
                indirectCost=indirectCost.replace("}]}","")
                Lists[2]=Lists[2]+int(indirectCost)
    return Lists

In [3]:

xlsx_path = Path("ShortMeeting history.xlsx")  # <-- change if needed

# Read sheets (your first sheet name is a bit odd, so grab by index)
xls = pd.ExcelFile(xlsx_path)
participants_df = pd.read_excel(xlsx_path, sheet_name=xls.sheet_names[0])
meetings_df      = pd.read_excel(xlsx_path, sheet_name="Meetings")

participants_df.iloc[ 1:2] = "Targeting Lipid Biology in Cancer"
participants_df.tail(5)

,Participant,Meeting,Type,First Name,Last Name,Suffix,Institution,Title
54,"Elsa Flores, PhD",2005 Scholar Retreat,Scholar,Elsa,Flores,PhD,MD Anderson Cancer Center,NaN
55,"Kimryn Rathmell, MD, PhD",2005 Scholar Retreat,Scholar,Kimryn,Rathmell,"MD, PhD",Vanderbilt University Medical Center,NaN
56,"Masashi Narita, MD, PhD",2005 Scholar Retreat,Scholar,Masashi,Narita,"MD, PhD",Cambridge Institute,NaN
57,"Jan Karlseder, PhD",2005 Scholar Retreat,Scholar,Jan,Karlseder,PhD,Salk Institute,NaN
58,"James Amatruda, MD, PhD",2005 Scholar Retreat,Scholar,James,Amatruda,"MD, PhD",Memorial Sloan Kettering Cancer Center,NaN


In [4]:
# Cell 2 — helpers (normalize meeting titles + parse chair names)

def normalize_meeting_title(x: str) -> str:
    """
    Make meeting titles comparable across sheets:
    - cast to str
    - strip leading/trailing whitespace
    - remove surrounding quotes
    - collapse internal whitespace (including newlines)
    """
    if pd.isna(x):
        return None
    s = str(x).strip()
    # remove one pair of surrounding quotes if present
    if (len(s) >= 2) and ((s[0] == s[-1]) and s[0] in {"'", '"'}):
        s = s[1:-1].strip()
    s = re.sub(r"\s+", " ", s)  # collapse newlines/tabs/multiple spaces
    return s

def split_chair_names(chairs_cell) -> list[str]:
    """
    Turn the 'Meeting Chairs' cell into a list of chair name strings.
    Handles separators like ';', ',', ' and ', '&', and common ' of ' patterns.
    Keeps credentials as part of the name string (e.g., 'MD, PhD').
    """
    if pd.isna(chairs_cell):
        return []
    s = str(chairs_cell).strip()
    s = re.sub(r"\s+", " ", s)

    # Many entries look like "Name of Institution; Name of Institution"
    # Split primarily on ';' first.
    parts = [p.strip() for p in s.split(";") if p.strip()]

    # Further split each part on " and " / " & " if it contains multiple chairs.
    chairs = []
    for p in parts:
        sub = re.split(r"\s+(?:and|&)\s+", p)
        for item in sub:
            item = item.strip()
            if not item:
                continue
            # Remove trailing institution phrase like " of XYZ" (optional, but helps matching)
            item = re.sub(r"\s+of\s+.+$", "", item).strip()
            chairs.append(item)

    # de-dup while preserving order
    seen = set()
    out = []
    for c in chairs:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out

In [5]:
meetings_df = meetings_df.copy()
participants_df = participants_df.copy()

meetings_df["MeetingTopic_norm"] = meetings_df["Meeting Topic"].map(normalize_meeting_title)
participants_df["Meeting_norm"]  = participants_df["Meeting"].map(normalize_meeting_title)

# Map normalized meeting topic -> year (if duplicates exist, keep the first non-null year)
meeting_to_year = (
    meetings_df.dropna(subset=["MeetingTopic_norm", "Year"])
               .drop_duplicates(subset=["MeetingTopic_norm"])
               .set_index("MeetingTopic_norm")["Year"]
               .to_dict()
)

# Attach year onto participants using normalized title
participants_df["Year"] = participants_df["Meeting_norm"].map(meeting_to_year)

participants_df[["Participant", "Meeting", "Year"]]

,Participant,Meeting,Year
0,"Alison Ringel, PhD",Targeting Lipid Biology in Cancer,2023
1,Targeting Lipid Biology in Cancer,Targeting Lipid Biology in Cancer,2023
2,"Bart Vanhaesebroeck, PhD",Targeting Lipid Biology in Cancer,2023
3,"Christina Mitchell, MB BS, PhD",Targeting Lipid Biology in Cancer,2023
4,"Neil Vasan, MD, PhD",Targeting Lipid Biology in Cancer,2023
5,"Prof Banafshe Larijani , PhD",Targeting Lipid Biology in Cancer,2023
6,"Ray Blind,",Targeting Lipid Biology in Cancer,2023
7,"Brooke Emerling, PhD",Targeting Lipid Biology in Cancer,2023
8,"Gretchen Alicea, PhD",Targeting Lipid Biology in Cancer,2023
9,"Sarah Skuli,",Targeting Lipid Biology in Cancer,2023


In [6]:
# Cell 4 — (1) create dict keyed by "Meeting Topic (Year)" with list of participant full names

# Keep only rows that have a meeting + participant name
p = participants_df.dropna(subset=["Meeting_norm", "Participant"]).copy()

# Build a key string like "Targeting Lipid Biology in Cancer (2021)"
def make_meeting_year_key(meeting_norm, year):
    y = "" if pd.isna(year) else str(int(year)) if float(year).is_integer() else str(year)
    return f"{meeting_norm} ({y})" if y else f"{meeting_norm} (Year Unknown)"

p["MeetingYearKey"] = [make_meeting_year_key(m, y) for m, y in zip(p["Meeting_norm"], p["Year"])]

meeting_attendees_dict = (
    p.groupby("MeetingYearKey")["Participant"]
     .apply(lambda s: sorted(set(s.dropna().astype(str).str.strip())))
     .to_dict()
)

# Example: show first 5 keys
list(meeting_attendees_dict["Targeting Lipid Biology in Cancer (2023)"])

['Alison Ringel, PhD',
 'Bart Vanhaesebroeck, PhD',
 'Brooke Emerling, PhD',
 'Christina Mitchell, MB BS, PhD',
 'David Fruman, PhD',
 'Emilio Hirsch, PhD',
 'Gretchen Alicea, PhD',
 'Hua Eleanor Yu, PhD',
 'Jeremy Baskin, PhD',
 'Karen Dixon,',
 'Livia  Schiavinato Eberlin, PhD',
 'Neil Vasan, MD, PhD',
 'Prof Banafshe  Larijani , PhD',
 'Ray Blind,',
 'Sarah  Skuli,',
 'Tamas Balla, MD, PhD',
 'Targeting Lipid Biology in Cancer',
 'Vytas Bankaitis, PhD']

In [9]:
REPORTER_SEARCH_URL = "https://api.reporter.nih.gov/v2/projects/search"
_SUFFIXES = {"jr","sr","ii","iii","iv","md","phd","mph","ms","m.d.","ph.d.","dr"}

def parse_first_last(name):
    if not name: return "", ""
    s = re.sub(r"\([^)]*\)", "", str(name))
    s = re.sub(r"^(dr\.?|prof\.?)\s+", "", s.strip(), flags=re.I)
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if len(parts) >= 2:
        last, first = parts[0], (parts[1].split()[0] if parts[1] else "")
        return first.lower(), last.lower()
    toks = [t for t in s.split() if t.lower().strip(".") not in _SUFFIXES]
    if len(toks) == 1: return "", toks[0].lower()
    return toks[0].lower(), toks[-1].lower()

def pi_entry(name):
    first, last = parse_first_last(name)
    if not last: return None
    return {"any_name": last, "first_name": first}

def meeting_year_from_key(meeting_key):
    m = re.search(r"\((\d{4})\)\s*$", str(meeting_key))
    return int(m.group(1)) if m else None

def five_year_bins(center_year, n_bins_before=1, n_bins_after=3):
    return ([(center_year-5*i, center_year-5*(i-1)) for i in range(n_bins_before, 0, -1)] +
            [(center_year+5*i, center_year+5*(i+1)) for i in range(0, n_bins_after)])

def fiscal_years_for_bin(start, end):
    return list(range(int(start), int(end) + 1))

def reporter_search_all_pages(payload, sleep=0.25, limit=500):
    params = deepcopy(payload); params.update({"offset": 0, "limit": limit})
    pages, total = [], None
    while True:
        r = requests.post(REPORTER_SEARCH_URL, json=params, timeout=60)
        r.raise_for_status()
        page = r.json(); pages.append(page)
        total = total or page.get("meta", {}).get("total", 0)
        off = page.get("meta", {}).get("offset", params["offset"])
        cnt = page.get("meta", {}).get("count", len(page.get("results", [])))
        if cnt == 0 or off + cnt >= total: break
        params["offset"] = off + cnt
        time.sleep(sleep)
    return {"total": int(total or 0), "pages": pages}

def multi_pi_collab_by_meeting(
    meeting_attendees_dict,
    NIH_param_template,
    n_bins_before=1,
    n_bins_after=3,
    sleep=0.25,
    print_every_n_pairs=200,   # set None to disable periodic progress prints
):
    summary_rows, detail_rows = [], []
    print("\n=== START MULTI-PI COLLAB ANALYSIS (DEDUP BY appl_id PER BIN) ===")

    for meeting_key, names in meeting_attendees_dict.items():
        print("\n===================================================")
        print(f"Meeting: {meeting_key}")
        print(f"Attendees in dict: {len(names)}")

        year = meeting_year_from_key(meeting_key)
        if year is None:
            print("  ⚠ No year found in meeting key; skipping.")
            continue
        print(f"Meeting year: {year}")

        clean = [(n, pi_entry(n)) for n in names]
        clean = [(n, e) for (n, e) in clean if e is not None]
        print(f"Parsable PI-like names: {len(clean)}")
        if len(clean) < 2:
            print("  ⚠ <2 parsable names; skipping meeting.")
            continue

        pairs = list(itertools.combinations(clean, 2))
        print(f"PI pairs to test: {len(pairs)}")

        for start, end in five_year_bins(year, n_bins_before, n_bins_after):
            print("\n-----------------------------------------------")
            print(f"Bin: {start}–{end}")
            fys = fiscal_years_for_bin(start, end)
            print(f"Fiscal years: {fys[0]}..{fys[-1]}")

            # Key change: dedupe at appl_id level per bin
            seen_appl_ids = set()
            unique_award_amount_sum = 0
            pair_hits = 0
            pair_idx = 0

            for (name_a, pi_a), (name_b, pi_b) in pairs:
                pair_idx += 1
                if print_every_n_pairs and pair_idx % print_every_n_pairs == 0:
                    print(f"  Progress: tested {pair_idx}/{len(pairs)} pairs; unique appl_ids so far: {len(seen_appl_ids)}")

                payload = deepcopy(NIH_param_template)
                payload.setdefault("criteria", {})
                c = payload["criteria"]
                c["pi_names"] = [pi_a, pi_b]
                c["multi_pi_only"] = True
                c["fiscal_years"] = fys
                c.pop("advanced_text_search", None)  # PI-only

                payload["include_fields"] = [
                    "appl_id","subproject_id","fiscal_year","project_num","project_title",
                    "fy_total_cost","award_amount","principal_investigators"
                ]

                res = reporter_search_all_pages(payload, sleep=sleep)
                if res["total"] <= 0:
                    continue

                pair_hits += 1
                pair_label = f"{name_a} || {name_b}"
                print(f"  Pair hit: {pair_label} -> {res['total']} records")

                # Record details, but mark whether the appl_id is first-seen in this bin
                for page in res["pages"]:
                    print("results")
                    print(page.get("results"))
                    for proj in page.get("results", []):
                        appl = proj.get("appl_id")
                        subp = proj.get("subproject_id")
                        fy = proj.get("fiscal_year")
                        pnum = proj.get("project_num")
                        title = proj.get("project_title")
                        cost = proj.get("fy_total_cost") if proj.get("fy_total_cost") is not None else proj.get("award_amount")
                        cost = int(cost or 0)

                        first_seen = appl not in seen_appl_ids
                        detail_rows.append({
                            "MeetingYearKey": meeting_key,
                            "Bin": f"{start}-{end}",
                            "Pair": pair_label,
                            "ApplId": appl,
                            "SubprojectId": subp,
                            "FiscalYear": fy,
                            "ProjectNum": pnum,
                            "ProjectTitle": title,
                            "AwardAmount": cost,
                            "IsFirstSeenInBin": first_seen,
                            "PrincipalInvestigators_raw": proj.get("principal_investigators")
                        })

                        # Dedup per bin: only count appl_id once
                        if first_seen:
                            seen_appl_ids.add(appl)
                            unique_award_amount_sum += cost

                time.sleep(sleep)

            print("\nBIN SUMMARY")
            print(f"  Pairs with ≥1 result: {pair_hits}")
            print(f"  Unique appl_ids in bin: {len(seen_appl_ids)}")
            print(f"  Summed award amount (unique appl_id only): ${unique_award_amount_sum:,}")

            summary_rows.append({
                "MeetingYearKey": meeting_key,
                "MeetingYear": year,
                "Bin": f"{start}-{end}",
                "Pairs_tested": len(pairs),
                "Pairs_with_hits": pair_hits,
                "Unique_appl_ids": len(seen_appl_ids),
                "Unique_award_amount_sum": unique_award_amount_sum
            })

    print("\n=== COMPLETE ===")
    return pd.DataFrame(summary_rows), pd.DataFrame(detail_rows)

In [ ]:
NIH_param={"criteria": {}}
NIH_param = {"criteria": {}}
summary_df, details_df = multi_pi_collab_by_meeting(
    meeting_attendees_dict,
    NIH_param_template=NIH_param,
    n_bins_before=1,
    n_bins_after=3,
    sleep=0.25,
    print_every_n_pairs=200
)
summary_df.sort_values(["MeetingYearKey","Bin"]).head(20)


=== START MULTI-PI COLLAB ANALYSIS (DEDUP BY appl_id PER BIN) ===

Meeting: 2005 Scholar Retreat (2005)
Attendees in dict: 17
Meeting year: 2005
Parsable PI-like names: 17
PI pairs to test: 136

-----------------------------------------------
Bin: 2000–2005
Fiscal years: 2000..2005
